# UFLD: conformal calibration and inference

Loads a pretrained causal Transformer and PPO patch selector, calibrates the conformal stopping thresholds on the calibration set (A/B procedure, Algorithms 2–4 in the paper) and evaluates adaptive acquisition on the test set.

Run `0-pretrain.ipynb` and `1-rl.ipynb` first.

In [ ]:
import os
import sys

# Run from the repository root so that `src/`, `data/` and `models/` resolve correctly
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [ ]:
import pickle

import numpy as np
import torch
from sklearn.metrics import f1_score

from src.config import MODEL_CONFIGS, classifier_checkpoint, ppo_model_name
from src.models.classifier.resnet import init_resnet
from src.models.patch_selection.classifier import CausalTransformerClassifierKVCached
from src.models.patch_selection.environment import WaferMapGrid
from src.models.patch_selection.ppo import PPOAgent
from src.utils.conformal import (
    make_AB_loaders,
    calibrate_multilabel_CP_two_rule_AB,
    evaluate_multilabel_CP_two_rule,
)
from src.utils.dataloader import load_mixedwm38_splits, get_loaders
from src.utils.utils import seed_everything

## Configuration

In [ ]:
SEED = 4
HYBRID = True          # fall back to ResNet-34 when the stopping rule never fires
MODEL_SIZE = 'large'   # must match the pretrained classifier and PPO agent
ALPHA = 0.065          # miscoverage level used for the main results

DATA_PATH = 'data/MixedWM38.npz'
RESULTS_DIR = 'results'

# Stopping-rule settings shared by calibration (set B) and evaluation
CP_KWARGS = dict(m=4, min_steps=1, patience=1, max_steps=None, break_ties_during_run=False)

seed_everything(seed=SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
os.makedirs(RESULTS_DIR, exist_ok=True)

dataset_name = 'mixedwm38'
classifier_name = 'transformer'
num_classes = 8
input_channels = 1
img_size = (52, 52)
batch_size = 128

## Data

In [ ]:
train_dataset, val_dataset, calib_dataset, test_dataset = load_mixedwm38_splits(
    DATA_PATH, img_size=img_size, input_channels=input_channels, num_classes=num_classes)

train_loader, test_loader, val_loader, calib_loader = get_loaders(
    train_dataset, test_dataset, val_dataset=val_dataset, calib_dataset=calib_dataset, batch_size=batch_size)

## Models

In [ ]:
embedding_dim, num_layers = MODEL_CONFIGS[MODEL_SIZE]

patch_size = 4
num_patches = (img_size[0] // patch_size) ** 2   # 169
hidden_dim = 2 * embedding_dim
num_heads = 4
num_output_heads = 64
c = 0.1

classifier = CausalTransformerClassifierKVCached(
    input_dim=patch_size ** 2,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    seq_len=num_patches,
    num_layers=num_layers,
    num_heads=num_heads,
    patch_size=patch_size,
    dropout_prob=0.1,
    batch_first=True,
    layer_norm=True,
    use_cls_token=False,
    embedding_dim=embedding_dim,
    use_patch_id_emb=True,
)

classifier.load_weights(classifier_checkpoint(dataset_name, SEED, num_layers, hidden_dim, num_heads=num_heads,
                                              num_output_heads=num_output_heads, c=c))
classifier = classifier.to(device)
classifier.eval()
classifier.freeze_all()

In [ ]:
reward_fun = "sparse"
env = WaferMapGrid(classifier, val_loader, test_loader, start_policy=0, reward_fun=reward_fun, device=device)

# lr, gamma, gae_lambda and SEED are part of the checkpoint file name
lr, gamma, gae_lambda = 1e-3, 0.99, 0.99
model_name = ppo_model_name(dataset_name, num_layers, hidden_dim, num_heads=num_heads,
                            num_output_heads=num_output_heads, c=c, reward_fun=reward_fun)

agent = PPOAgent(n_actions=env.action_space.n, obs_dim=classifier.embed_dim,
                 batch_size=batch_size, gamma=gamma, alpha=lr, gae_lambda=gae_lambda,
                 n_epochs=100, policy_clip=0.2, seed=SEED, chkpt_dir='models', model_name=model_name, device=device)
agent.load_models()

## Conformal calibration (A/B)

In [ ]:
calib_loader_A, calib_loader_B = make_AB_loaders(
    calib_dataset, batch_size=batch_size, split=0.5, shuffle=True, num_workers=4, pin_memory=True)

q0_A, q1_A, q0_B, q1_B = calibrate_multilabel_CP_two_rule_AB(
    agent=agent, env=env,
    calib_loader_A=calib_loader_A, calib_loader_B=calib_loader_B,
    alpha=ALPHA, device=device, **CP_KWARGS,
)

## Test evaluation

In [ ]:
if HYBRID:
    cnn_model = init_resnet('resnet34', num_classes=num_classes, input_channels=input_channels, pretrained=False)
    cnn_model.load_state_dict(torch.load(f"models/{dataset_name}/resnet34_seed{SEED}.pth"))
    cnn_model = cnn_model.to(device)
    cnn_model.eval()
else:
    cnn_model = None

In [ ]:
test_results = evaluate_multilabel_CP_two_rule(
    agent=agent, env=env, test_loader=test_loader,
    q0=q0_B, q1=q1_B, cnn_model=cnn_model, device=device, **CP_KWARGS,
)

with open(os.path.join(RESULTS_DIR, f'final-res_seed{SEED}.pickle'), 'wb') as handle:
    pickle.dump(test_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Per-defect accuracy and F1 score (Figure 3)
defect_names = ["Center", "Donut", "Edge_Loc", "Edge_Ring", "Loc", "Near_Full", "Scratch", "Random"]
Y_true, Y_pred = test_results['Y_true'], test_results['Y_pred']

acc_by_defect = (Y_true == Y_pred).mean(axis=0)
f1_by_defect = np.array([f1_score(Y_true[:, i], Y_pred[:, i]) for i in range(num_classes)])

for name, acc, f1 in zip(defect_names, acc_by_defect, f1_by_defect):
    print(f"{name:10s}  acc = {acc:.4f}  F1 = {f1:.4f}")

## Sensitivity to the miscoverage level (Table 5)

Run once with `HYBRID = False`, then again with `HYBRID = True`: the hybrid run loads the non-hybrid results and adds the hybrid accuracy.

In [ ]:
alphas = np.linspace(0.05, 0.1, 11)
sweep_file = os.path.join(
    RESULTS_DIR,
    f"ppo_bs16_{dataset_name}_c{c}_supcon{classifier_name}_sch_r-{reward_fun}0.005_{num_output_heads}outs_{num_heads}heads_{num_layers}layers{hidden_dim}_clip0.5_seed{SEED}.pickle")

if HYBRID:
    with open(sweep_file, 'rb') as f:
        all_results = pickle.load(f)
    all_results["hybrid_subset_acc"] = []
else:
    all_results = {"alpha": [], "subset_acc": [], "coverage": [], "avg_steps": []}

for a in alphas:
    print(f"alpha = {a}")
    _, _, q0_B, q1_B = calibrate_multilabel_CP_two_rule_AB(
        agent=agent, env=env,
        calib_loader_A=calib_loader_A, calib_loader_B=calib_loader_B,
        alpha=a, device=device, **CP_KWARGS,
    )
    res = evaluate_multilabel_CP_two_rule(
        agent=agent, env=env, test_loader=test_loader,
        q0=q0_B, q1=q1_B, cnn_model=cnn_model, device=device, **CP_KWARGS,
    )

    if HYBRID:
        all_results["hybrid_subset_acc"].append(res["subset_acc"])
    else:
        all_results["alpha"].append(a)
        all_results["subset_acc"].append(res["subset_acc"])
        all_results["coverage"].append(res["coverage"])
        all_results["avg_steps"].append(res["avg_steps"])

with open(sweep_file, 'wb') as handle:
    pickle.dump(all_results, handle, protocol=pickle.HIGHEST_PROTOCOL)